# 数据加载

In [1]:
#import kagglehub

# Download latest version
#path = kagglehub.competition_download('word2vec-nlp-tutorial')

#print("Path to competition files:", path)

# 数据解压

In [2]:
# 解压数据
import glob
import os
import zipfile

# 压缩包所在的文件夹路径与解压目标路径
zip_dir = "/kaggle/input/competitions/word2vec-nlp-tutorial"  # 存放多个 zip 的文件夹
output_dir = "/kaggle/working/"  # 解压出来的目标文件夹

# 获取所有 .zip 文件的路径
zip_files = glob.glob(os.path.join(zip_dir, "*.zip"))

for file_path in zip_files:
    print(f"正在解压: {file_path}")
    with zipfile.ZipFile(file_path, "r") as zip_ref:
        # 解压到统一的目标文件夹（如果想按文件名单独建文件夹，可用下面的方法）
        zip_ref.extractall(output_dir)

print("所有 ZIP 文件解压完成！")

os.makedirs("./result", exist_ok=True)

正在解压: /kaggle/input/competitions/word2vec-nlp-tutorial/testData.tsv.zip
正在解压: /kaggle/input/competitions/word2vec-nlp-tutorial/unlabeledTrainData.tsv.zip
正在解压: /kaggle/input/competitions/word2vec-nlp-tutorial/labeledTrainData.tsv.zip
所有 ZIP 文件解压完成！


In [3]:
pip install evaluate

Note: you may need to restart the kernel to use updated packages.


In [4]:
!pip install unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.2/30.2 MB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 945.8/945.8 kB 23.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 66.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 34.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 38.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 39.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 38.7 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [unsloth]


# imdb-modernbert-unsloth

In [5]:
import os

os.environ["HF_ENDPOINT"] = "https://huggingface.co"
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

import unsloth
import torch
import sys
import logging
import evaluate

import pandas as pd
import numpy as np

from unsloth import FastModel, FastLanguageModel
from transformers import TrainingArguments, Trainer, AutoModelForSequenceClassification, training_args, \
    DataCollatorWithPadding
from datasets import Dataset

from sklearn.model_selection import train_test_split

train = pd.read_csv("/kaggle/working/labeledTrainData.tsv", header=0, delimiter="\t", quoting=3)
test = pd.read_csv("/kaggle/working/testData.tsv", header=0, delimiter="\t", quoting=3)

if __name__ == '__main__':
    program = os.path.basename(sys.argv[0])
    logger = logging.getLogger(program)

    logging.basicConfig(format='%(asctime)s: %(levelname)s: %(message)s')
    logging.root.setLevel(level=logging.INFO)
    logger.info(r"running %s" % ''.join(sys.argv))

    train, val = train_test_split(train, test_size=.2)

    #train = train[0:20]
    #val = val[0:20]
    #test = test[0:20]

    train_dict = {'label': train["sentiment"], 'text': train['review']}
    val_dict = {'label': val["sentiment"], 'text': val['review']}
    test_dict = {"text": test['review']}

    train_dataset = Dataset.from_dict(train_dict)
    val_dataset = Dataset.from_dict(val_dict)
    test_dataset = Dataset.from_dict(test_dict)

    # model_name = 'answerdotai/ModernBERT-large'
    model_name = "microsoft/deberta-v2-xxlarge"
    NUM_CLASSES = 2

    model, tokenizer = FastModel.from_pretrained(
        model_name=model_name,
        load_in_4bit=False,
        max_seq_length=2048,
        dtype=torch.bfloat16,
        auto_model=AutoModelForSequenceClassification,
        num_labels=NUM_CLASSES,
        gpu_memory_utilization=0.8  # Reduce if out of memory
    )

    model = FastModel.get_peft_model(
        model,
        r=16,  # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
        # target_modules=[
        #     "q_proj", "k_proj", "v_proj", "o_proj",
        #     "gate_proj", "up_proj", "down_proj", ],
        lora_alpha=32,
        lora_dropout=0,  # Supports any, but = 0 is optimized
        bias="none",  # Supports any, but = "none" is optimized
        # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
        use_gradient_checkpointing="unsloth",  # True or "unsloth" for very long context
        random_state=3407,
        use_rslora=False,  # We support rank stabilized LoRA
        loftq_config=None,  # And LoftQ
        task_type="SEQ_CLS",
    )

    print("model parameters:" + str(sum(p.numel() for p in model.parameters())))

    # make all parameters trainable
    # for param in model.parameters():
    #     param.requires_grad = True

    metric = evaluate.load("accuracy")


    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return metric.compute(predictions=predictions, references=labels)


    def tokenize_function(examples):
        # return tokenizer(examples['text'])
        return tokenizer(examples['text'], max_length=512, truncation=True)


    train_dataset = train_dataset.map(tokenize_function, batched=True)
    val_dataset = val_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    training_args = TrainingArguments(
        per_device_train_batch_size=32,
        gradient_accumulation_steps=1,

        warmup_steps=10,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim=training_args.OptimizerNames.ADAMW_TORCH,
        learning_rate=2e-5,
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        num_train_epochs=1,  # bert-style models usually need more than 1 epoch
        save_strategy="epoch",

        # report_to="wandb",

        # group_by_length=True,
        # eval_strategy="no",
        eval_strategy="no",
        # eval_steps=0.25,
        logging_strategy="steps",
        logging_steps=10,

        label_names=["label"],
    )

    from typing import Any, Dict, Tuple, Optional
    class UnslothSafeTrainer(Trainer):
        """
        安全的 prediction_step：**不** 调用 model.prediction_step（避免 unsloth 的 monkey-patch）。
        直接使用 model(**inputs) 并提取 loss/logits/labels。
        """

        def prediction_step(
                self,
                model,
                inputs: Dict[str, Any],
                prediction_loss_only: bool,
                ignore_keys: Optional[Tuple[str]] = None,
        ):
            # 把 inputs 放到设备上（复用 Trainer 的内部方法）
            inputs = self._prepare_inputs(inputs)

            # 获取是否有 labels（有时 key 叫 labels，有时叫 label）
            labels = None
            if isinstance(inputs, dict):
                labels = inputs.get("labels", inputs.get("label", None))

            # 直接用 forward（不要调用 model.prediction_step）
            with torch.no_grad():
                outputs = model(**inputs)

            # 解析 outputs：支持 dict 或 tuple
            loss = None
            logits = None

            # 如果 model 返回 dict（huggingface 习惯），尽量取标准字段
            if isinstance(outputs, dict):
                # loss 可能在 outputs['loss']
                loss = outputs.get("loss", None)
                # logits 可能在 outputs['logits'] 或 outputs['predictions']
                logits = outputs.get("logits", outputs.get("predictions", None))
            elif isinstance(outputs, tuple):
                # tuple 常见格式: (logits, ...) 或 (loss, logits, ...)
                if len(outputs) == 0:
                    logits = None
                elif len(outputs) == 1:
                    logits = outputs[0]
                elif len(outputs) == 2:
                    # 一般 (loss, logits) 或 (logits, labels) —— 尝试智能判断
                    a, b = outputs[0], outputs[1]
                    # 若 a 是标量 tensor 且 b 不是标量，猜 a 是 loss
                    if getattr(a, "ndim", None) == 0 or (isinstance(a, torch.Tensor) and a.numel() == 1):
                        loss = a
                        logits = b
                    else:
                        logits = a
                else:
                    # 常见 (loss, logits, ...) 或 (logits, ...)
                    # 优先把第一个标量当 loss，第二个当 logits
                    if getattr(outputs[0], "ndim", None) == 0 or (
                            isinstance(outputs[0], torch.Tensor) and outputs[0].numel() == 1):
                        loss = outputs[0]
                        logits = outputs[1] if len(outputs) > 1 else None
                    else:
                        logits = outputs[0]

            # 有时候 loss 在 outputs.loss，但未转到 cpu，保持 tensor
            if prediction_loss_only:
                # 只返回 loss 情况
                return (loss, None, None)

            # 确保 logits、labels 都在 cpu/device 上与 Trainer 期望一致
            return (loss, logits, labels)

    trainer = UnslothSafeTrainer(
        model=model,
        args=training_args,
        processing_class=tokenizer,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer_stats = trainer.train()

    print(trainer_stats)

    # model.eval()
    # FastLanguageModel.for_inference(model)

    prediction_outputs = trainer.predict(test_dataset)
    print(prediction_outputs)
    test_pred = np.argmax(prediction_outputs[0], axis=-1).flatten()
    print(test_pred)

    result_output = pd.DataFrame(data={"id": test["id"], "sentiment": test_pred})
    result_output.to_csv("./result/deberta_unsloth.csv", index=False, quoting=3)
    logging.info('result saved!')


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


[colab_kernel_launcher.py|INFO]running /usr/local/lib/python3.13/dist-packages/colab_kernel_launcher.py-f/tmp/tmpf9krrf3_.json--HistoryManager.hist_file=:memory:
[unsloth_zoo.log|WARNING]Device does not support bfloat16. Will change to float16.
[httpx|INFO]HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v2-xxlarge/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[httpx|INFO]HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v2-xxlarge/38f115b79f551ca4091c9caef41940fd1d6312b9/config.json "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: GET https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v2-xxlarge/38f115b79f551ca4091c9caef41940fd1d6312b9/config.json "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v2-xxlarge/resolve/main/adapter_config.json "HTTP/1.1 404 Not Found"


==((====))==  Unsloth 2026.10.11: Fast Deberta_V2 patching. Transformers: 5.16.1.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


[httpx|INFO]HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/revision/main "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: GET https://huggingface.co/api/models/unslothai/repeat/revision/main "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: GET https://huggingface.co/api/models/unslothai/1/revision/main "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: GET https://huggingface.co/api/models/unslothai/kaggle/tree/b632e7c464e861a6f1762dd396048ab1ed7a10ec?recursive=true&expand=false "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: GET https://huggingface.co/api/models/unslothai/vram-16/revision/main "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: GET https://huggingface.co/api/models/unslothai/repeat/tree/7c48478c02f84ed89f149b0815cc0216ee831fb0?recursive=true&expand=false "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: HEAD https://huggingface.co/unslothai/kaggle/resolve/b632e7c464e861a6f1762dd396048ab1ed7a10ec/README.md "HTTP/1.1 307 Temporary Redirect"
[httpx|INFO]HTTP Request: HEAD https://hug

Unsloth: Not planning a device map; the load builds DebertaV2ForSequenceClassification, not the planned DebertaV2Model. Using `sequential`.


...

Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


[httpx|INFO]HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v2-xxlarge/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[httpx|INFO]HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v2-xxlarge/38f115b79f551ca4091c9caef41940fd1d6312b9/config.json "HTTP/1.1 200 OK"
[httpx|INFO]HTTP Request: HEAD https://huggingface.co/microsoft/deberta-v2-xxlarge/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
[httpx|INFO]HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/microsoft/deberta-v2-xxlarge/38f115b79f551ca4091c9caef41940fd1d6312b9/config.json "HTTP/1.1 200 OK"
Unsloth: Restored added_tokens_decoder metadata in trainer_output/checkpoint-625/tokenizer_config.json.
[httpx|INFO]HTTP Request: GET https://huggingface.co/api/models/microsoft/deberta-v2-xxlarge "HTTP/1.1 200 OK"


TrainOutput(global_step=625, training_loss=0.31359913983345034, metrics={'train_runtime': 13981.3402, 'train_samples_per_second': 1.43, 'train_steps_per_second': 0.045, 'total_flos': 8.464843765560806e+16, 'train_loss': 0.31359913983345034, 'epoch': 1.0})


[root|INFO]result saved!


PredictionOutput(predictions=array([[-2.8652344,  2.203125 ],
       [ 3.359375 , -2.6171875],
       [ 1.7607422, -1.1308594],
       ...,
       [ 2.6757812, -2.2734375],
       [-2.9667969,  2.5527344],
       [-2.3144531,  1.5634766]], dtype=float32), label_ids=None, metrics={'test_runtime': 4643.2062, 'test_samples_per_second': 5.384, 'test_steps_per_second': 0.673})
[1 0 0 ... 0 1 1]
